<a href="https://colab.research.google.com/github/Dado233242/intro-ai-/blob/main/lab3_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 3

## Задание
Решите пожалуйста следующую задачу.

1. Примените методы снижения размерности (до двумерного представления
n_components=2) перечисленные ниже для данных находящихся пo
[ссылке](https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view). Здесь - метка (учитель) является столбец / атрибут - `outcome`. Предоставьте результаты, используя следующие ядерные функции:
- PCA
- LDA
- TruncatedSVD
- Kernel PCA (перебрав все ядерные функции - ('linear', 'poly', 'rbf', sigmoid',
'cosine', 'precomputed')

## Загрузка данных

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics.pairwise import rbf_kernel

In [23]:
! gdown --fuzzy 'https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view'

Downloading...
From: https://drive.google.com/uc?id=1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec
To: /content/diabetes.csv
100% 23.1k/23.1k [00:00<00:00, 41.4MB/s]


In [24]:
df = pd.read_csv('/content/diabetes.csv')
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


##  О датасете
[Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database/data)

Этот набор данных взят из Национального института диабета, заболеваний органов пищеварения и почек (NIDDK). Целью набора данных является диагностическое предсказание наличия у пациента диабета на основе определенных диагностических измерений, включенных в набор данных. На выбор этих экземпляров из более крупной базы данных было наложено несколько ограничений. В частности, все пациенты здесь — женщины не моложе 21 года, родом из индейцев Пима.

Наборы данных состоят из нескольких переменных медицинских предикторов и одной целевой переменной `Outcome`. Переменные-предикторы включают количество беременностей, которые были у пациентки, её индекс массы тела `BMI`, уровень инсулина, возраст и так далее. Всего в датасете 9 переменных:
- `Pregnancies` - количество беременностей;
- `Glucose` - концентрация глюкозы в плазме через 2 часа при пероральном тесте на толерантность к глюкозе;
- `BloodPressure` - диастолическое артериальное давление (мм рт. ст.);
- `SkinThickness` - толщина кожной складки трицепса (мм);
- `Insulin` - 2-часовой сывороточный инсулин (мкЕд/мл);
- `BMI` - Индекс массы тела (вес в кг/(рост в м)^2);
- `DiabetesPedigreeFunction` - функция оценки вероятности диабета на основе семейного анамнеза;
- `Age` - возраст (лет);
- `Outcome` - целевая переменная (0 или 1) 268 из 768 — 1 (больны), остальные — 0 (здоровы).



In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [26]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [27]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [28]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = {}

## Метод PCA

In [29]:
pca = PCA(n_components=2, random_state=42)
results["PCA"] = pca.fit_transform(X_scaled)
print("PCA - Variance expliquée :", np.round(pca.explained_variance_ratio_, 4))
print("PCA - Variance cumulée   :", round(np.sum(pca.explained_variance_ratio_), 4))


PCA - Variance expliquée : [0.2618 0.2164]
PCA - Variance cumulée   : 0.4782


## Метод LDA

In [30]:
lda = LinearDiscriminantAnalysis(n_components=1)
results["LDA (1 composante)"] = lda.fit_transform(X_scaled, y)
print("LDA - Variance expliquée :", np.round(lda.explained_variance_ratio_, 4))

LDA - Variance expliquée : [1.]


## Метод TruncatedSVD

In [31]:
# n_components=2
svd = TruncatedSVD(n_components=2, random_state=42)
results["TruncatedSVD"] = svd.fit_transform(X_scaled)
print("SVD - explained_variance :", np.round(svd.explained_variance_ratio_, 4))

SVD - explained_variance : [0.2618 0.2164]


## Метод Kernel PCA

###  Ядерная функция `linear`

In [32]:
kpca_linear = KernelPCA(n_components=2, kernel="linear", random_state=42)
X_linear = kpca_linear.fit_transform(X_scaled)
print("Linear - Premières coordonnées :\n", X_linear[:3])

Linear - Premières coordonnées :
 [[ 1.06850273  1.23489499]
 [-1.12168331 -0.73385167]
 [-0.39647671  1.59587594]]


###  Ядерная функция `poly`

In [33]:
kpca_poly = KernelPCA(n_components=2, kernel="poly", degree=3, random_state=42)
X_poly = kpca_poly.fit_transform(X_scaled)
print("Poly - Premières coordonnées :\n", X_poly[:3])

Poly - Premières coordonnées :
 [[-0.13601502 -0.56637788]
 [-0.46648679 -0.02000078]
 [-0.38446888  0.00104629]]


###  Ядерная функция `rbf`

In [34]:
kpca_rbf = KernelPCA(n_components=2, kernel="rbf", random_state=42)
X_rbf = kpca_rbf.fit_transform(X_scaled)
print("RBF - Premières coordonnées :\n", X_rbf[:3])

RBF - Premières coordonnées :
 [[-0.38562027  0.08508466]
 [ 0.4750535  -0.08933368]
 [-0.30144303 -0.21057116]]


###  Ядерная функция `sigmoid`

In [35]:
kpca_sigmoid = KernelPCA(n_components=2, kernel="sigmoid", random_state=42)
X_sigmoid = kpca_sigmoid.fit_transform(X_scaled)
print("Sigmoid - Premières coordonnées :\n", X_sigmoid[:3])

Sigmoid - Premières coordonnées :
 [[-0.26657115  0.26687044]
 [ 0.25768669 -0.16880087]
 [ 0.09025704  0.39666942]]


###  Ядерная функция `cosine`

In [36]:
kpca_cosine = KernelPCA(n_components=2, kernel="cosine", random_state=42)
X_cosine = kpca_cosine.fit_transform(X_scaled)
print("Cosine - Premières coordonnées :\n", X_cosine[:3])

Cosine - Premières coordonnées :
 [[ 0.83252829  0.09349974]
 [-0.67196458 -0.14922513]
 [ 0.44899913 -0.47600931]]


###  Ядерная функция `precomputed`

In [37]:
from sklearn.metrics.pairwise import rbf_kernel

#Calcul manuel de la matrice de Gram
K_custom = rbf_kernel(X_scaled)

#Passage de la matrice précalculée à KernelPCA
kpca_pre = KernelPCA(n_components=2, kernel="precomputed", random_state=42)
X_pre = kpca_pre.fit_transform(K_custom)
print("Precomputed - Premières coordonnées :\n", X_pre[:3])

Precomputed - Premières coordonnées :
 [[-0.38562027  0.08508466]
 [ 0.4750535  -0.08933368]
 [-0.30144303 -0.21057116]]
